# <div style="text-align: center">Gmail Classification Models</div>

**Import Libraries**

In [ ]:
import pickle
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

import nltk
from nltk.tokenize import word_tokenize
from nltk.tokenize import RegexpTokenizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


**Read Excel file**

In [ ]:
#df = pd.read_excel(r'Downloads\All.xlsx')
df = pd.read_csv('dataset\mail_data.csv')
df.head()

In [ ]:
#thêm cột label number với ham 0,spam 1
df['Label_Number'] = df['Category']
df.loc[df['Label_Number']=='ham','Label_Number',] = 0
df.loc[df['Label_Number']=='spam','Label_Number',] = 1
df.head()
pickle.dump(df,open("models/df_raw.pkl","wb"))

In [ ]:
df.shape #kích thước tập dữ liệu

In [ ]:
df.info()

In [ ]:
df.isna().sum() #kiểm tra giá trị thiếu

In [ ]:
df['Category'].value_counts() #số lượng dữ liệu với mỗi nhãn

**Count Plot - vẽ biểu đồ thể hiện số lượng dữ liệu**

In [ ]:
plt.figure(figsize = (8, 6))
sns.countplot(data = df, x = 'Category');

**Đếm số lượng từ của mỗi dòng trong trường Text**

In [ ]:
def count_words(text):
    words = word_tokenize(text)
    return len(words)
df['count']=df['Message'].apply(count_words)
df.head()
#df['count']

In [ ]:
df.groupby('Category').mean()

**Tokenization**

In [ ]:
%%time
def clean_str(string, reg = RegexpTokenizer(r'[a-z]+')):
    # Clean a string with RegexpTokenizer
    string = string.lower()
    tokens = reg.tokenize(string)
    return " ".join(tokens)

print('Trước khi cleaning:')
df.head()

In [ ]:
print('Sau cleaning:')
df['Message'] = df['Message'].apply(lambda string: clean_str(string))
df.head()

**Stemming words - loại bỏ 1 số ký tự nằm ở cuối từ**

In [ ]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
def stemming (text):
    return ''.join([stemmer.stem(word) for word in text])
df['Message']=df['Message'].apply(stemming)
df.head()
pickle.dump(df,open("models/df_train.pkl","wb"))


In [ ]:
X = df.loc[:, 'Message']
y = df.loc[:, 'Label_Number']

print(f"Kích thước tập đầu vào X: {X.shape}\nKích thước tập đầu ra y: {y.shape}")

**Chia dữ liệu thành Training data và Test data dùng Split**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=11)

y_train = y_train.astype('int')
y_test = y_test.astype('int')

In [ ]:
print(f"Kích thước Training Data : {X_train.shape}\nKích thước Test Data: {X_test.shape}")

**Dùng Count Vectorization trích rút Features từ Text**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

cv1=CountVectorizer()
cv1.fit(X_train)

cv2= TfidfVectorizer(min_df = 1)
cv2.fit(X_train)#Học tạo ra danh sách từ vừng

In [ ]:
print('Số lượng từ vựng CountVectorrizer: ',len(cv1.vocabulary_.keys()))
print('Số lượng từ vựng TfidfVectorizer: ',len(cv2.vocabulary_.keys()))

In [ ]:
dtv1 = cv1.transform(X_train) #chuyển đổi tài liệu thành ma trận document-term
dtv2 = cv2.transform(X_train)
#type(dtv)
#lưu cv
pickle.dump(cv1, open("vectorizer1.pkl","wb"))
pickle.dump(cv2, open("vectorizer2.pkl","wb"))

In [ ]:
#biến đổi X_train
dtv1 = dtv1.toarray()
dtv2 = dtv2.toarray()
#biến đổi X_test
test_dtv1 = cv1.transform(X_test)
test_dtv1 = test_dtv1.toarray()
test_dtv2 = cv2.transform(X_test)
test_dtv2 = test_dtv2.toarray()

#dtv

In [ ]:
#trích rút đặc trưng bằng CountVectorizer
print(f"Số dòng dữ liệu đầu vào để Train: {dtv1.shape[0]}\nSố lượng Features: {dtv1.shape[1]}")
print(f"Số dòng dữ liệu để test: {test_dtv1.shape[0]}\nSố lượng Features tập X test: {test_dtv1.shape[1]}")
#trích rút đặc trưng bằng TfidVectorizer
print(f"Số dòng dữ liệu đầu vào để Train: {dtv2.shape[0]}\nSố lượng Features: {dtv2.shape[1]}")
print(f"Số dòng dữ liệu để test: {test_dtv2.shape[0]}\nSố lượng Features tập X test: {test_dtv2.shape[1]}")

**Áp dụng 7 thuật toán tạo model cho tập Train **

In [ ]:
%%time

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC, SVC
from time import perf_counter
from sklearn.metrics import precision_score,recall_score,f1_score,accuracy_score,roc_auc_score
import warnings
warnings.filterwarnings(action='ignore')


#CountVectorizer
models1 = {
    "Random Forest": {"model":RandomForestClassifier(), "perf":0},
    "MultinomialNB": {"model":MultinomialNB(), "perf":0},
    "Logistic Regression": {"model":LogisticRegression(solver='liblinear', penalty ='l2' , C = 1.0), "perf":0},
    "KNN": {"model":KNeighborsClassifier(), "perf":0},
    "Decision Tree": {"model":DecisionTreeClassifier(), "perf":0},
    "SVM (Linear)": {"model":LinearSVC(), "perf":0},
    "SVM (RBF)": {"model":SVC(), "perf":0}
}
#TfidfVectorizer
models2 = {
    "Random Forest": {"model":RandomForestClassifier(), "perf":0},
    "MultinomialNB": {"model":MultinomialNB(), "perf":0},
    "Logistic Regression": {"model":LogisticRegression(solver='liblinear', penalty ='l2' , C = 1.0), "perf":0},
    "KNN": {"model":KNeighborsClassifier(), "perf":0},
    "Decision Tree": {"model":DecisionTreeClassifier(), "perf":0},
    "SVM (Linear)": {"model":LinearSVC(), "perf":0},
    "SVM (RBF)": {"model":SVC(), "perf":0}
}
#CountVectorrizer
for name, model in models1.items():
    start = perf_counter()
    model['model'].fit(dtv1, y_train)
    duration = perf_counter() - start #tính giờ train
    duration = round(duration,3)#làm tròn
    model["perf"] = duration
    print(f"{name:20} trained in {duration} sec")
    pickle.dump(model['model'], open("models/CountVectorizer/"+str(name)+".pkl","wb"))

In [ ]:
%%time
#TfidfVectorizer
for name, model in models2.items():
    start = perf_counter()
    model['model'].fit(dtv2, y_train)
    duration = perf_counter() - start #tính giờ train
    duration = round(duration,3)#làm tròn
    model["perf"] = duration
    print(f"{name:20} trained in {duration} sec")
    pickle.dump(model['model'], open("models/TfidfVectorizer/"+str(name)+".pkl","wb"))


**Test Accuracy and Training Time**

In [ ]:
#tính độ chính xác, tính các độ đo khác cho 7 thuật toán
def acc(models,test_dtv):
    models_accuracy = []
    metrics_models = []
    for name, model in models.items():
        pre = model["model"].predict(test_dtv)
        models_accuracy.append([name, accuracy_score(y_test,pre)*100,model["perf"]])
        metrics_models.append([name,accuracy_score(y_test,pre)*100,
                                   precision_score(y_test,pre)*100,
                                   recall_score(y_test,pre)*100,
                                   f1_score(y_test,pre)*100 ])
    return models_accuracy,metrics_models
#in ra bảng độ chính xác
def dfacc(models_accuracy):
    df_accuracy = pd.DataFrame(models_accuracy)
    df_accuracy.columns = ['Model', 'Test Accuracy', 'Training time (sec)']
    df_accuracy.reset_index(drop = True, inplace=True)
    return df_accuracy

In [ ]:
print("Trích rút đặc trưng với CountVectorizer")
models_accuracy1 = []
metrics_models1 = []
models_accuracy1,metrics_models1 = acc(models1,test_dtv1)
#gọi hàm in ra bảng
df_accuracy1 = dfacc(models_accuracy1)
pickle.dump(df_accuracy1,open("models/table_acc_count.pkl","wb"))
df_accuracy1


In [ ]:
print("Trích rút đặc trưng với TfidfVectorizer")
models_accuracy2 = []
metrics_models2 = []
models_accuracy2,metrics_models2 = acc(models2,test_dtv2)
df_accuracy2 = dfacc(models_accuracy2)
pickle.dump(df_accuracy2,open("models/table_acc_tfidf.pkl","wb"))
df_accuracy2

In [ ]:
def plotacc(df_accuracy,name_plot):
    plt.figure(figsize = (15,5))
    sns.barplot(x = 'Model', y ='Test Accuracy', data = df_accuracy)
    plt.title('Accuracy on the test set\n', fontsize = 15)
    plt.ylim(90,100)
    plt.savefig("img_models/"+str(name_plot),dpi=150)
    plt.show()
print("Mô hình - Trích rút đặc trưng với CountVectorizer")
plotacc(df_accuracy1,"CountVectorizer/plot_acc_count.png")
print("Mô hình - Trích rút đặc trưng với TfidfVectorizer")
plotacc(df_accuracy2,"TfidfVectorizer/plot_acc_tfidf.png")

In [ ]:
def plottime(df_accuracy,name_plot):
    plt.figure(figsize = (15,5))
    sns.barplot(x = 'Model', y = 'Training time (sec)', data = df_accuracy)
    plt.title('Training time for each model in sec', fontsize = 15)
    plt.ylim(0,45)
    plt.savefig("img_models/"+str(name_plot),dpi=150)
    plt.show()
print("Mô hình - Trích rút đặc trưng với CountVectorizer")
plottime(df_accuracy1,"CountVectorizer/plot_time_count.png")
print("Mô hình - Trích rút đặc trưng với TfidfVectorizer")
plottime(df_accuracy2,"TfidfVectorizer/plot_time_tfidf.png")

In [ ]:
#in ra bảng độ đo khác
def dfmetrics(metrics_models,name_table):
    df_metrics = pd.DataFrame(metrics_models)
    df_metrics.columns = ['Model', 'Accuracy score', 'Precision score','Recall score','F1 score']
    df_metrics.reset_index(drop = True, inplace=True)
    pickle.dump(df_metrics,open("models/"+str(name_table)+".pkl","wb"))
    return df_metrics

In [ ]:
print("Mô hình - Trích rút đặc trưng với CountVectorizer")
dfmetrics(metrics_models1,"table_metrics_count")

In [ ]:
print("Mô hình - Trích rút đặc trưng với TfidfVectorizer")
dfmetrics(metrics_models2,"table_metrics_tfidf")

## **Logistic Regression**<br>


In [ ]:
from sklearn.metrics import plot_confusion_matrix, plot_roc_curve, plot_precision_recall_curve

model1 = models1["Logistic Regression"]
model2 = models2["Logistic Regression"]

In [ ]:
#ma trận nhầm lẫn
plot_confusion_matrix(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/Logistic Regression/cfm.png',dpi=120)
plot_confusion_matrix(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/Logistic Regression/cfm.png',dpi=120)
#Biểu đồ ROC
plot_roc_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/Logistic Regression/roc.png',dpi=120)
plot_roc_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/Logistic Regression/roc.png',dpi=120)
#Biểu đồ precision recall
plot_precision_recall_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/Logistic Regression/precision_recall.png',dpi=120)
plot_precision_recall_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/Logistic Regression/precision_recall.png',dpi=120)


## **Support Vector Machine (RBF)**<br>

In [ ]:
model1 = models1["SVM (RBF)"]
model2 = models2["SVM (RBF)"]

In [ ]:
#ma trận nhầm lẫn
plot_confusion_matrix(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/SVM (RBF)/cfm.png',dpi=120)
plot_confusion_matrix(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/SVM (RBF)/cfm.png',dpi=120)
#Biểu đồ ROC
plot_roc_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/SVM (RBF)/roc.png',dpi=120)
plot_roc_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/SVM (RBF)/roc.png',dpi=120)
#Biểu đồ precision recall
plot_precision_recall_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/SVM (RBF)/precision_recall.png',dpi=120)
plot_precision_recall_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/SVM (RBF)/precision_recall.png',dpi=120)

## **Random Forest Classifier**<br>

In [ ]:
model1 = models1["Random Forest"]
model2 = models2["Random Forest"]

In [ ]:
#ma trận nhầm lẫn
plot_confusion_matrix(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/Random Forest/cfm.png',dpi=120)
plot_confusion_matrix(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/Random Forest/cfm.png',dpi=120)
#Biểu đồ ROC
plot_roc_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/Random Forest/roc.png',dpi=120)
plot_roc_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/Random Forest/roc.png',dpi=120)
#Biểu đồ precision recall
plot_precision_recall_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/Random Forest/precision_recall.png',dpi=120)
plot_precision_recall_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/Random Forest/precision_recall.png',dpi=120)


## **Multinomial Naive Bayes** <br>

In [ ]:
model1 = models1["MultinomialNB"]
model2 = models2["MultinomialNB"]

In [ ]:
#ma trận nhầm lẫn
plot_confusion_matrix(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/MultinomialNB/cfm.png',dpi=120)
plot_confusion_matrix(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/MultinomialNB/cfm.png',dpi=120)
#Biểu đồ ROC
plot_roc_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/MultinomialNB/roc.png',dpi=120)
plot_roc_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/MultinomialNB/roc.png',dpi=120)
#Biểu đồ precision recall
plot_precision_recall_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/MultinomialNB/precision_recall.png',dpi=120)
plot_precision_recall_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/MultinomialNB/precision_recall.png',dpi=120)


## **Support Vector Machine (Linear)** <br>

In [ ]:
model1 = models1["SVM (Linear)"]
model2 = models2["SVM (Linear)"]

In [ ]:
#ma trận nhầm lẫn
plot_confusion_matrix(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/SVM (Linear)/cfm.png',dpi=120)
plot_confusion_matrix(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/SVM (Linear)/cfm.png',dpi=120)
#Biểu đồ ROC
plot_roc_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/SVM (Linear)/roc.png',dpi=120)
plot_roc_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/SVM (Linear)/roc.png',dpi=120)
#Biểu đồ precision recall
plot_precision_recall_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/SVM (Linear)/precision_recall.png',dpi=120)
plot_precision_recall_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/SVM (Linear)/precision_recall.png',dpi=120)


## **Decision Tree Classifier** <br>

In [ ]:
model1 = models1["Decision Tree"]
model2 = models2["Decision Tree"]

In [ ]:
#ma trận nhầm lẫn
plot_confusion_matrix(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/Decision Tree/cfm.png',dpi=120)
plot_confusion_matrix(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/Decision Tree/cfm.png',dpi=120)
#Biểu đồ ROC
plot_roc_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/Decision Tree/roc.png',dpi=120)
plot_roc_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/Decision Tree/roc.png',dpi=120)
#Biểu đồ precision recall
plot_precision_recall_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/Decision Tree/precision_recall.png',dpi=120)
plot_precision_recall_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/Decision Tree/precision_recall.png',dpi=120)


## **K Nearest Neighbours**<br>

In [ ]:
model1 = models1["KNN"]
model2 = models2["KNN"]

In [ ]:
#ma trận nhầm lẫn
plot_confusion_matrix(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/KNN/cfm.png',dpi=120)
plot_confusion_matrix(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/KNN/cfm.png',dpi=120)
#Biểu đồ ROC
plot_roc_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/KNN/roc.png',dpi=120)
plot_roc_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/KNN/roc.png',dpi=120)
#Biểu đồ precision recall
plot_precision_recall_curve(model1["model"], test_dtv1, y_test)
plt.savefig('img_models/CountVectorizer/KNN/precision_recall.png',dpi=120)
plot_precision_recall_curve(model2["model"], test_dtv2, y_test)
plt.savefig('img_models/TfidfVectorizer/KNN/precision_recall.png',dpi=120)


